## TLDR
* approved borrowers get assigned an interest rate
* offered a loan at that interest rate
* loan is listed on marketplace (along with borrower info)
* loan is funded by an investor
* borrower gets $ and begins paying back loan

## Lending Club step-by-step process
* borrower is given a grade
* aka Lending Club evaluates each borrower's credit score using past historical data and their own data science process
* interest rate: is the percent & requested loan to be paid back: interest rates assigned: https://www.lendingclub.com/public/borrower-rates-and-fees.action
* interest rates range from 5.32% all the way to 30.99%
* borrower is  givena grade according to the interest rate they were assigned: grade: https://www.lendingclub.com/public/rates-and-fees.action
* borrower is offered a loan amoutn
* borrower accepts interest rate
* loan is listed on the Lending Club marketplace
* info is listed along with loan
* if loan is funded, borrower receives $ minus Lending Club origination fee


https://www.lendingclub.com/

https://en.wikipedia.org/wiki/Credit_risk

supposed to be Lending Club's data for approved & declined loans:  https://www.lendingclub.com/public/how-peer-lending-works.action
* link redirects to main page


origination fee that Lending Club charges:  https://help.lendingclub.com/hc/en-us/articles/214501207-What-is-the-origination-fee-

## LCDataDictionary.xlsx
### column dictionary
https://docs.google.com/spreadsheets/d/191B2yJ4H1ZPXq0_ByhUgWMFZOYem5jFz0Y3by_7YBY4

## Goals
* become familiar with the columns in the dataset
* select the target column
* decide on the type of model
* remove many columns that aren't useful for modeling

In [ ]:
import pandas as pd
loans_2007 = pd.read_csv("loans_2007.csv")

print(loans_2007.head(1))
print(loans_2007.info())

        id  member_id  loan_amnt  funded_amnt  funded_amnt_inv        term  \
0  1077501  1296599.0     5000.0       5000.0           4975.0   36 months   

  int_rate  installment grade sub_grade  ... last_pymnt_amnt  \
0   10.65%       162.87     B        B2  ...          171.62   

  last_credit_pull_d collections_12_mths_ex_med  policy_code application_type  \
0           Jun-2016                        0.0          1.0       INDIVIDUAL   

  acc_now_delinq chargeoff_within_12_mths delinq_amnt pub_rec_bankruptcies  \
0            0.0                      0.0         0.0                  0.0   

  tax_liens  
0       0.0  

[1 rows x 52 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25925 entries, 0 to 25924
Data columns (total 52 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          25925 non-null  int64  
 1   member_id                   25925 non-null  float64
 2   l

In [ ]:
print(loans_2007.columns)

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title',
       'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
       'issue_d', 'loan_status', 'pymnt_plan', 'purpose', 'title', 'zip_code',
       'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line',
       'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv',
       'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
       'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
       'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d',
       'collections_12_mths_ex_med', 'policy_code', 'application_type',
       'acc_now_delinq', 'chargeoff_within_12_mths', 'delinq_amnt',
       'pub_rec_bankruptcies', 'tax_liens'],
      dtype='object')



look at column descriptions

look for any features that:

* disclose information from the future (after the loan has already been funded)
don't affect a borrower's ability to pay back a loan (e.g. a randomly generated ID value by Lending Club)
* need to be cleaned up and are formatted poorly
* require more data or a lot of processing to turn into a useful feature
* contain redundant information

## First 18 Columns to consider removing

| name                | dtype   | first value | description                                                                                                                       |
|---------------------|---------|-------------|-----------------------------------------------------------------------------------------------------------------------------------|
| id                  | object  | 1077501     | A unique LC assigned ID for the loan listing.                                                                                     |
| member_id           | float64 | 1.2966e+06  | A unique LC assigned Id for the borrower member.                                                                                  |
| loan_amnt           | float64 | 5000        | The listed amount of the loan applied for by the borrower.                                                                        |
| funded_amnt         | float64 | 5000        | The total amount committed to that loan at that point in time.                                                                    |
| funded_amnt_inv     | float64 | 49750       | The total amount committed by investors for that loan at that point in time.                                                      |
| term                | object  | 36 months   | The number of payments on the loan. Values are in months and can be either 36 or 60.                                              |
| int_rate            | object  | 10.65%      | Interest Rate on the loan                                                                                                         |
| installment         | float64 | 162.87      | The monthly payment owed by the borrower if the loan originates.                                                                  |
| grade               | object  | B           | LC assigned loan grade                                                                                                            |
| sub_grade           | object  | B2          | LC assigned loan subgrade                                                                                                         |
| emp_title           | object  | NaN         | The job title supplied by the Borrower when applying for the loan.                                                                |
| emp_length          | object  | 10+ years   | Employment length in years. Possible values are between 0 and 10 where 0 means less than one year and 10 means ten or more years. |
| home_ownership      | object  | RENT        | The home ownership status provided by the borrower during registration. Our values are: RENT, OWN, MORTGAGE, OTHER.               |
| annual_inc          | float64 | 24000       | The self-reported annual income provided by the borrower during registration.                                                     |
| verification_status | object  | Verified    | Indicates if income was verified by LC, not verified, or if the income source was verified                                        |
| issue_d             | object  | Dec-2011    | The month which the loan was funded                                                                                               |
| loan_status         | object  | Charged Off | Current status of the loan                                                                                                        |
| pymnt_plan          | object  | n           | Indicates if a payment plan has been put in place for the loan                                                                    |
| purpose             | object  | car         | A category provided by the borrower for the loan request.                                                                         |


## Columns removed
* **id**: randomly generated field by Lending Club for unique identification purposes only
* **member_id**: also a randomly generated field by Lending Club for unique identification purposes only
* **funded_amnt**: leaks data from the future (after the loan is already started to be funded)
* **funded_amnt_inv**: also leaks data from the future (after the loan is already started to be funded)
* **grade**: contains redundant information as the interest rate column (int_rate)
* **sub_grade**: also contains redundant information as the interest rate column (int_rate)
* **emp_title**: requires other data and a lot of processing to potentially be useful
* **issue_d**: leaks data from the future (after the loan is already completely funded)

In [ ]:
drop_cols = ['id', 'member_id', 'funded_amnt', 'funded_amnt_inv', 'grade', 'sub_grade', 'emp_title', 'issue_d']
loans_2007 = loans_2007.drop(columns=drop_cols)

## 2nd 18 Columns to consider removing
| name                | dtype   | first value | description                                                                                                                                                                                              |
|---------------------|---------|-------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| title               | object  | Computer    | The loan title provided by the borrower                                                                                                                                                                  |
| zip_code            | object  | 860xx       | The first 3 numbers of the zip code provided by the borrower in the loan application.                                                                                                                    |
| addr_state          | object  | AZ          | The state provided by the borrower in the loan application                                                                                                                                               |
| dti                 | float64 | 27.65       | A ratio calculated using the borrower’s total monthly debt payments on the total debt obligations, excluding mortgage and the requested LC loan, divided by the borrower’s self-reported monthly income. |
| delinq_2yrs         | float64 | 0           | The number of 30+ days past-due incidences of delinquency in the borrower's credit file for the past 2 years                                                                                             |
| earliest_cr_line    | object  | Jan-1985    | The month the borrower's earliest reported credit line was opened                                                                                                                                        |
| inq_last_6mths      | float64 | 1           | The number of inquiries in past 6 months (excluding auto and mortgage inquiries)                                                                                                                         |
| open_acc            | float64 | 3           | The number of open credit lines in the borrower's credit file.                                                                                                                                           |
| pub_rec             | float64 | 0           | Number of derogatory public records                                                                                                                                                                      |
| revol_bal           | float64 | 13648       | Total credit revolving balance                                                                                                                                                                           |
| revol_util          | object  | 83.7%       | Revolving line utilization rate, or the amount of credit the borrower is using relative to all available revolving credit.                                                                               |
| total_acc           | float64 | 9           | The total number of credit lines currently in the borrower's credit file                                                                                                                                 |
| initial_list_status | object  | f           | The initial listing status of the loan. Possible values are – W, F                                                                                                                                       |
| out_prncp           | float64 | 0           | Remaining outstanding principal for total amount funded                                                                                                                                                  |
| out_prncp_inv       | float64 | 0           | Remaining outstanding principal for portion of total amount funded by investors                                                                                                                          |
| total_pymnt         | float64 | 5863.16     | Payments received to date for total amount funded                                                                                                                                                        |
| total_pymnt_inv     | float64 | 5833.84     | Payments received to date for portion of total amount funded by investors                                                                                                                                |
| total_rec_prncp     | float64 | 5000        | Principal received to date                                                                                                                                                                               |


## Columns removed
* **zip_code**: redundant with the addr_state column since only the first 3 digits of the 5-digit zip code are visible (which can only be used to identify the state the borrower lives in)
* **out_prncp**: leaks data from the future, (after the loan already started to be paid off)
* **out_prncp_inv**: also leaks data from the future, (after the loan already started to be paid off)
* **total_pymnt**: also leaks data from the future, (after the loan already started to be paid off)
* **total_pymnt_inv**: also leaks data from the future, (after the loan already started to be paid off)
* **total_rec_prncp**: also leaks data from the future, (after the loan already started to be paid off)

In [ ]:
drop_cols = ['zip_code', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp']
loans_2007 = loans_2007.drop(columns=drop_cols)

## Final Columns to consider removing
| name                       | dtype   | first value | description                                                                                          |
|----------------------------|---------|-------------|------------------------------------------------------------------------------------------------------|
| total_rec_int              | float64 | 863.16      | Interest received to date                                                                            |
| total_rec_late_fee         | float64 | 0           | Late fees received to date                                                                           |
| recoveries                 | float64 | 0           | post charge off gross recovery                                                                       |
| collection_recovery_fee    | float64 | 0           | post charge off collection fee                                                                       |
| last_pymnt_d               | object  | Jan-2015    | Last month payment was received                                                                      |
| last_pymnt_amnt            | float64 | 171.62      | Last total payment amount received                                                                   |
| last_credit_pull_d         | object  | Jun-2016    | The most recent month LC pulled credit for this loan                                                 |
| collections_12_mths_ex_med | float64 | 0           | Number of collections in 12 months excluding medical collections                                     |
| policy_code                | float64 | 1           | publicly available policy_code=1 new products not publicly available policy_code=2                   |
| application_type           | object  | INDIVIDUAL  | Indicates whether the loan is an individual application or a joint application with two co-borrowers |
| acc_now_delinq             | float64 | 0           | The number of accounts on which the borrower is now delinquent.                                      |
| chargeoff_within_12_mths   | float64 | 0           | Number of charge-offs within 12 months                                                               |
| delinq_amnt                | float64 | 0           | The past-due amount owed for the accounts on which the borrower is now delinquent.                   |
| pub_rec_bankruptcies       | float64 | 0           | Number of public record bankruptcies                                                                 |
| tax_liens                  | float64 | 0           | Number of tax liens                                                                                  |


## Columns removed
* **total_rec_int**: leaks data from the future, (after the loan has started to be paid off),
* **total_rec_late_fee**: leaks data from the future, (after the loan has started to be paid off),
* **recoveries**: leaks data from the future, (after the loan has started to be paid off),
* **collection_recovery_fee**: leaks data from the future, (after the loan has started to be paid off),
* **last_pymnt_d**: leaks data from the future, (after the loan has started to be paid off),
* **last_pymnt_amnt**: leaks data from the future, (after the loan has started to be paid off).

In [ ]:
drop_cols = ['total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt']
loans_2007 = loans_2007.drop(columns=drop_cols)

## Feature Categories for Target Column
Since we're trying to predict whether a loan is either Fully Paid or Not, it should be a binary classification model. Thus all other 'loan_status' categories (other than 'Fully Paid' & 'Charged Off') are irrelevant to the model. Those other categories pertain to the ongoing nature of the loan.

In [ ]:
# analyze the 'loan_status' column as a potential target column
# see what distribution of values there
# consider mapping the valuus to categorical values
print(loans_2007.loan_status.value_counts())

Fully Paid            21091
Charged Off            3818
Current                 961
Late (31-120 days)       24
In Grace Period          20
Late (16-30 days)         8
Default                   3
Name: loan_status, dtype: int64


## Loan Status column value explainations
| Loan Status                                         | Count | Meaning                                                                                                                                           |
|-----------------------------------------------------|-------|---------------------------------------------------------------------------------------------------------------------------------------------------|
| Fully Paid                                          | 33136 | Loan has been fully paid off.                                                                                                                     |
| Charged Off                                         | 5634  | Loan for which there is no longer a reasonable expectation of further payments.                                                                   |
| Does not meet the credit policy. Status:Fully Paid  | 1988  | While the loan was paid off, the loan application today would no longer meet the credit policy and wouldn't be approved on to the marketplace.    |
| Does not meet the credit policy. Status:Charged Off | 761   | While the loan was charged off, the loan application today would no longer meet the credit policy and wouldn't be approved on to the marketplace. |
| In Grace Period                                     | 20    | The loan is past due but still in the grace period of 15 days.                                                                                    |
| Late (16-30 days)                                   | 8     | Loan hasn't been paid in 16 to 30 days (late on the current payment).                                                                             |
| Late (31-120 days)                                  | 24    | Loan hasn't been paid in 31 to 120 days (late on the current payment).                                                                            |
| Current                                             | 961   | Loan is up to date on current payments.                                                                                                           |
| Default                                             | 3     | Loan is defaulted on and no payment has been made for more than 121 days.                                                                         |


In [ ]:
# filter out useless categories
keep_filter = (loans_2007['loan_status']=="Fully Paid") | (loans_2007['loan_status']=="Charged Off")
loans_2007 = loans_2007[keep_filter]

# print(keep_filter)
print(loans_2007[keep_filter]['loan_status'].value_counts())

Fully Paid     21091
Charged Off     3818
Name: loan_status, dtype: int64


/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:6: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  


In [ ]:
# convert the loan categorical classifications to numerical categories
loan_status_mapping = {
    "Fully Paid":1,
    "Charged Off":0
}

loans_2007['loan_status'] = loans_2007['loan_status'].replace(loan_status_mapping)
print(loans_2007[keep_filter]['loan_status'].value_counts())

1    21091
0     3818
Name: loan_status, dtype: int64


/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:8: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  


In [ ]:
# Removing Single Value Columns
# columns with 0 variance convey no information the model

drop_columns = []
for c in loans_2007.columns:
    non_null = loans_2007[c].dropna()
    unique_non_null = non_null.unique()
    if len(unique_non_null) == 1:
        drop_columns.append(c)

loans_2007 = loans_2007.drop(columns=drop_columns)
print(drop_columns)

['pymnt_plan', 'initial_list_status', 'collections_12_mths_ex_med', 'policy_code', 'application_type', 'acc_now_delinq', 'chargeoff_within_12_mths', 'delinq_amnt', 'tax_liens']


In [ ]:
loans_2007.to_csv("filtered_loans_2007.csv")